In [23]:
# =========================
# CELL 1 — IMPORTS + CONFIG
# =========================
from google.colab import drive
drive.mount('/content/drive')

import os
import json
import random
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score,
    roc_curve,
    auc
)
from sklearn.preprocessing import label_binarize

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# =========================
# PATH CONFIG
# =========================
PROJECT_DIR = "/content/drive/MyDrive/S-Class/Orion/OrionFL/orion-fl-dr-research"
DATASET_NAME = "DDR"
EXPERIMENT_NAME = "exp1_baseline"
MODEL_NAME = "mobilenet"

DDR_ROOT_DIR = "/content/drive/MyDrive/S-Class/Orion/OrionFL/DDR_Dataset"
IMAGE_DIR = os.path.join(DDR_ROOT_DIR, "DR_grading")
CSV_PATH = os.path.join(DDR_ROOT_DIR, "DR_grading.csv")

BASE_RESULT_DIR = os.path.join(PROJECT_DIR, "results", DATASET_NAME, EXPERIMENT_NAME, MODEL_NAME)
MODELS_DIR = os.path.join(BASE_RESULT_DIR, "models")
LOGS_DIR = os.path.join(BASE_RESULT_DIR, "logs")
FIGURES_DIR = os.path.join(BASE_RESULT_DIR, "figures")

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(LOGS_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

# =========================
# TRAINING CONFIG
# =========================
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

EPOCHS_STAGE1 = 20
EPOCHS_STAGE2 = 20

LEARNING_RATE_STAGE1 = 1e-3
LEARNING_RATE_STAGE2 = 1e-6

OPTIMIZER_STAGE1 = "Adam"
OPTIMIZER_STAGE2 = "Adam"
LOSS_FUNCTION = "SparseCategoricalCrossentropy"

UNFREEZE_LAST_N_LAYERS = 10

# default, nanti dioverwrite setelah CSV dibaca
NUM_CLASSES = None

print("TensorFlow version:", tf.__version__)
print("PROJECT_DIR:", PROJECT_DIR)
print("DDR_ROOT_DIR:", DDR_ROOT_DIR)
print("IMAGE_DIR:", IMAGE_DIR, "| exists:", os.path.exists(IMAGE_DIR))
print("CSV_PATH:", CSV_PATH, "| exists:", os.path.exists(CSV_PATH))
print("BASE_RESULT_DIR:", BASE_RESULT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
TensorFlow version: 2.19.0
PROJECT_DIR: /content/drive/MyDrive/S-Class/Orion/OrionFL/orion-fl-dr-research
DDR_ROOT_DIR: /content/drive/MyDrive/S-Class/Orion/OrionFL/DDR_Dataset
IMAGE_DIR: /content/drive/MyDrive/S-Class/Orion/OrionFL/DDR_Dataset/DR_grading | exists: True
CSV_PATH: /content/drive/MyDrive/S-Class/Orion/OrionFL/DDR_Dataset/DR_grading.csv | exists: True
BASE_RESULT_DIR: /content/drive/MyDrive/S-Class/Orion/OrionFL/orion-fl-dr-research/results/DDR/exp1_baseline/mobilenet


In [24]:
# =========================
# CELL 2 — CHECK DIRECTORY CONTENT
# =========================
if os.path.exists(DDR_ROOT_DIR):
    print("Files/folders inside DDR_ROOT_DIR:")
    for f in os.listdir(DDR_ROOT_DIR):
        print("-", f)
else:
    print("DDR_ROOT_DIR not found.")

Files/folders inside DDR_ROOT_DIR:
- DR_grading.csv
- DR_grading
- DDR_dataset_224.npz


In [25]:
# =========================
# CELL 3 — LOAD CSV + QUICK INSPECTION
# =========================
assert os.path.exists(CSV_PATH), f"CSV file tidak ditemukan: {CSV_PATH}"

df_raw = pd.read_csv(CSV_PATH)

print("CSV shape:", df_raw.shape)
print("CSV columns:", list(df_raw.columns))
display(df_raw.head())

CSV shape: (12522, 2)
CSV columns: ['id_code', 'diagnosis']


,id_code,diagnosis
0,20170413102628830.jpg,0
1,20170413111955404.jpg,0
2,20170413112015395.jpg,0
3,20170413112017305.jpg,0
4,20170413112528859.jpg,0


In [26]:
# =========================
# CELL 4 — SET COLUMNS MANUALLY
# =========================
filename_col = "id_code"   # ganti sesuai nama kolom file yang muncul
label_col = "diagnosis"

print("Selected filename column:", filename_col)
print("Selected label column:", label_col)

assert filename_col in df_raw.columns, f"Kolom filename '{filename_col}' tidak ada."
assert label_col in df_raw.columns, f"Kolom label '{label_col}' tidak ada."

Selected filename column: id_code
Selected label column: diagnosis


In [27]:
# =========================
# CELL 5 — INDEX ALL IMAGE FILES
# =========================
VALID_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

all_image_paths = []
for root, _, files in os.walk(IMAGE_DIR):
    for f in files:
        ext = Path(f).suffix.lower()
        if ext in VALID_EXTS:
            all_image_paths.append(os.path.join(root, f))

print("Total image files found:", len(all_image_paths))

# map basename tanpa extension -> full path
image_map = {}
for p in all_image_paths:
    stem = Path(p).stem
    image_map[stem] = p

# juga map basename lengkap kalau CSV sudah ada extension
image_map_full = {}
for p in all_image_paths:
    name = Path(p).name
    image_map_full[name] = p

Total image files found: 12524


In [28]:
# =========================
# CELL 6 — BUILD CLEAN DATAFRAME (PATH + LABEL)
# =========================
df = df_raw.copy()

df[filename_col] = df[filename_col].astype(str).str.strip()
df[label_col] = pd.to_numeric(df[label_col], errors="coerce")

# drop label NaN
df = df.dropna(subset=[label_col]).copy()
df[label_col] = df[label_col].astype(int)

def resolve_image_path(x):
    x = str(x).strip()
    if x in image_map_full:
        return image_map_full[x]

    stem = Path(x).stem
    if stem in image_map:
        return image_map[stem]

    return None

df["image_path"] = df[filename_col].apply(resolve_image_path)

missing_df = df[df["image_path"].isna()].copy()
matched_df = df[df["image_path"].notna()].copy()

print("Rows in CSV:", len(df))
print("Matched rows:", len(matched_df))
print("Missing rows:", len(missing_df))

if len(missing_df) > 0:
    print("\nSample missing rows:")
    display(missing_df[[filename_col, label_col]].head())

matched_df = matched_df[[filename_col, label_col, "image_path"]].copy()
matched_df = matched_df.rename(columns={label_col: "label", filename_col: "image_id"})

matched_df = matched_df.drop_duplicates(subset=["image_path"]).reset_index(drop=True)

print("\nFinal usable dataframe:", matched_df.shape)
display(matched_df.head())

Rows in CSV: 12522
Matched rows: 12522
Missing rows: 0

Final usable dataframe: (12522, 3)


,image_id,label,image_path
0,20170413102628830.jpg,0,/content/drive/MyDrive/S-Class/Orion/OrionFL/D...
1,20170413111955404.jpg,0,/content/drive/MyDrive/S-Class/Orion/OrionFL/D...
2,20170413112015395.jpg,0,/content/drive/MyDrive/S-Class/Orion/OrionFL/D...
3,20170413112017305.jpg,0,/content/drive/MyDrive/S-Class/Orion/OrionFL/D...
4,20170413112528859.jpg,0,/content/drive/MyDrive/S-Class/Orion/OrionFL/D...


In [29]:
# =========================
# CELL 7 — BASIC DATA SUMMARY
# =========================
label_counts = matched_df["label"].value_counts().sort_index()
NUM_CLASSES = int(matched_df["label"].nunique())

print("NUM_CLASSES:", NUM_CLASSES)
print("\nOverall label distribution:")
print(label_counts)

summary = {
    "dataset_name": DATASET_NAME,
    "num_samples": int(len(matched_df)),
    "num_classes": int(NUM_CLASSES),
    "label_distribution": {str(int(k)): int(v) for k, v in label_counts.to_dict().items()},
    "image_dir": IMAGE_DIR,
    "csv_path": CSV_PATH,
    "filename_column": filename_col,
    "label_column": label_col
}

summary_path = os.path.join(LOGS_DIR, "dataset_summary.json")
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=4)

print("\nSaved:", summary_path)

NUM_CLASSES: 5

Overall label distribution:
label
0    6266
1     630
2    4477
3     236
4     913
Name: count, dtype: int64

Saved: /content/drive/MyDrive/S-Class/Orion/OrionFL/orion-fl-dr-research/results/DDR/exp1_baseline/mobilenet/logs/dataset_summary.json


In [30]:
# =========================
# CELL 8 — GLOBAL TRAIN/VAL/TEST SPLIT (70:15:15)
# =========================
train_df, temp_df = train_test_split(
    matched_df,
    test_size=0.30,
    random_state=SEED,
    stratify=matched_df["label"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["label"]
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Train shape:", train_df.shape)
print("Val shape  :", val_df.shape)
print("Test shape :", test_df.shape)

print("\nTrain label distribution:")
print(train_df["label"].value_counts().sort_index())

print("\nVal label distribution:")
print(val_df["label"].value_counts().sort_index())

print("\nTest label distribution:")
print(test_df["label"].value_counts().sort_index())

Train shape: (8765, 3)
Val shape  : (1878, 3)
Test shape : (1879, 3)

Train label distribution:
label
0    4386
1     441
2    3134
3     165
4     639
Name: count, dtype: int64

Val label distribution:
label
0    940
1     94
2    671
3     36
4    137
Name: count, dtype: int64

Test label distribution:
label
0    940
1     95
2    672
3     35
4    137
Name: count, dtype: int64


In [31]:
# =========================
# CELL 9 — COMPUTE CLASS WEIGHTS
# =========================
classes = np.sort(train_df["label"].unique())

class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=train_df["label"].values
)

class_weights = {int(cls): float(w) for cls, w in zip(classes, class_weights_array)}

print("Class weights:")
print(class_weights)

Class weights:
{0: 0.3996808025535796, 1: 3.9750566893424035, 2: 0.5593490746649649, 3: 10.624242424242425, 4: 2.7433489827856024}


In [32]:
# =========================
# CELL 10 — BUILD TF DATASETS + LIGHT AUGMENTATION
# =========================
AUTOTUNE = tf.data.AUTOTUNE

data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.03),
    layers.RandomZoom(0.05),
], name="light_augmentation")

def load_and_preprocess_image(path, label):
    image = tf.io.read_file(path)

    def _decode_image(img_bytes, path_str):
        path_str = path_str.numpy().decode("utf-8").lower()
        if path_str.endswith(".png"):
            img = tf.image.decode_png(img_bytes, channels=3)
        elif path_str.endswith(".bmp"):
            img = tf.image.decode_bmp(img_bytes)
        else:
            img = tf.image.decode_jpeg(img_bytes, channels=3)
        return img

    image = tf.py_function(_decode_image, [image, path], Tout=tf.uint8)
    image.set_shape([None, None, 3])

    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32)
    image = preprocess_input(image)

    label = tf.cast(label, tf.int32)
    return image, label

def augment_fn(x, y):
    x = data_augmentation(x, training=True)
    return x, y

train_ds = tf.data.Dataset.from_tensor_slices(
    (train_df["image_path"].values, train_df["label"].values)
)
train_ds = train_ds.shuffle(buffer_size=len(train_df), seed=SEED)
train_ds = train_ds.map(load_and_preprocess_image, num_parallel_calls=AUTOTUNE)
train_ds = train_ds.batch(BATCH_SIZE)
train_ds = train_ds.map(augment_fn, num_parallel_calls=AUTOTUNE)
train_ds = train_ds.prefetch(AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices(
    (val_df["image_path"].values, val_df["label"].values)
)
val_ds = val_ds.map(load_and_preprocess_image, num_parallel_calls=AUTOTUNE)
val_ds = val_ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

test_ds = tf.data.Dataset.from_tensor_slices(
    (test_df["image_path"].values, test_df["label"].values)
)
test_ds = test_ds.map(load_and_preprocess_image, num_parallel_calls=AUTOTUNE)
test_ds = test_ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

print("train_ds ready with light augmentation")
print("val_ds ready")
print("test_ds ready")

train_ds ready with light augmentation
val_ds ready
test_ds ready


In [33]:
# =========================
# CELL 11 — DEFINE MODEL BUILDER
# =========================
def build_mobilenetv2_model(input_shape=(224, 224, 3), num_classes=5):
    base_model = MobileNetV2(
        include_top=False,
        weights="imagenet",
        input_shape=input_shape
    )
    base_model.trainable = False

    inputs = keras.Input(shape=input_shape)
    x = base_model(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(
        num_classes,
        activation="softmax",
        kernel_regularizer=regularizers.l2(1e-4)
    )(x)

    model = keras.Model(inputs, outputs)
    return model, base_model

In [34]:
# =========================
# CELL 12 — BUILD MODEL
# =========================
model, base_model = build_mobilenetv2_model(
    input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3),
    num_classes=NUM_CLASSES
)

model.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_5 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 5)              │         6,405 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,264,389 (8.64 MB)

 Trainable params: 6,405 (25.02 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [35]:
# =========================
# CELL 13 — COMPILE STAGE 1
# =========================
STAGE1_BEST_PATH = os.path.join(MODELS_DIR, "best_mobilenetv2_stage1.keras")

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE_STAGE1),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"]
)

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=2,
        min_lr=1e-7,
        verbose=1
    ),
    keras.callbacks.ModelCheckpoint(
        filepath=STAGE1_BEST_PATH,
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    )
]

print("Stage 1 compile done.")

Stage 1 compile done.


In [ ]:
# =========================
# CELL 14 — TRAIN STAGE 1
# =========================
history_stage1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_STAGE1,
    class_weight=class_weights,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/20
  7/274 ━━━━━━━━━━━━━━━━━━━━ 38:43 9s/step - accuracy: 0.1505 - loss: 1.9298

In [ ]:
# =========================
# CELL 15 — UNFREEZE LAST LAYERS FOR SAFE FINE-TUNING
# =========================
base_model.trainable = True

for layer in base_model.layers[:-UNFREEZE_LAST_N_LAYERS]:
    layer.trainable = False

for layer in base_model.layers[-UNFREEZE_LAST_N_LAYERS:]:
    layer.trainable = True

for layer in base_model.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

trainable_count = sum([1 for layer in base_model.layers if layer.trainable])
print(f"Trainable layers in base_model: {trainable_count} / {len(base_model.layers)}")

In [ ]:
# =========================
# CELL 16 — COMPILE STAGE 2
# =========================
STAGE2_BEST_PATH = os.path.join(MODELS_DIR, "best_mobilenetv2_finetune.keras")

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE_STAGE2),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"]
)

callbacks_stage2 = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=2,
        min_lr=1e-7,
        verbose=1
    ),
    keras.callbacks.ModelCheckpoint(
        filepath=STAGE2_BEST_PATH,
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    )
]

print("Stage 2 compile done.")

In [ ]:
# =========================
# CELL 17 — TRAIN STAGE 2
# =========================
history_stage2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_STAGE2,
    class_weight=class_weights,
    callbacks=callbacks_stage2,
    verbose=1
)

In [ ]:
# =========================
# CELL 18 — PLOT TRAINING HISTORY + SAVE HISTORY JSON
# =========================
def to_serializable_history(history_obj):
    return {k: [float(x) for x in v] for k, v in history_obj.history.items()}

def plot_history(histories, save_path=None):
    if not isinstance(histories, list):
        histories = [histories]

    all_acc = []
    all_val_acc = []
    all_loss = []
    all_val_loss = []

    for hist in histories:
        all_acc.extend(hist.history.get("accuracy", []))
        all_val_acc.extend(hist.history.get("val_accuracy", []))
        all_loss.extend(hist.history.get("loss", []))
        all_val_loss.extend(hist.history.get("val_loss", []))

    plt.figure(figsize=(14, 5))

    plt.subplot(1, 2, 1)
    plt.plot(all_acc, label="train_acc")
    plt.plot(all_val_acc, label="val_acc")
    plt.title("Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(all_loss, label="train_loss")
    plt.plot(all_val_loss, label="val_loss")
    plt.title("Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()

    plt.tight_layout()

    if save_path is not None:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")

    plt.show()
    plt.close()

history_stage1_path = os.path.join(LOGS_DIR, "history_stage1.json")
history_stage2_path = os.path.join(LOGS_DIR, "history_stage2.json")
history_combined_path = os.path.join(LOGS_DIR, "history_combined.json")

history_stage1_dict = to_serializable_history(history_stage1)
history_stage2_dict = to_serializable_history(history_stage2)

with open(history_stage1_path, "w") as f:
    json.dump(history_stage1_dict, f, indent=4)

with open(history_stage2_path, "w") as f:
    json.dump(history_stage2_dict, f, indent=4)

history_combined = {
    "stage1": history_stage1_dict,
    "stage2": history_stage2_dict
}

with open(history_combined_path, "w") as f:
    json.dump(history_combined, f, indent=4)

history_plot_path = os.path.join(FIGURES_DIR, "training_history.png")
plot_history([history_stage1, history_stage2], save_path=history_plot_path)

print("Saved:", history_stage1_path)
print("Saved:", history_stage2_path)
print("Saved:", history_combined_path)
print("Saved:", history_plot_path)

In [ ]:
# =========================
# CELL 19 — FINAL EVALUATION ON TEST SET
# =========================
test_probs = model.predict(test_ds, verbose=1)
test_preds = np.argmax(test_probs, axis=1)

y_true = test_df["label"].values
y_pred = test_preds
y_prob = test_probs

report_text = classification_report(y_true, y_pred, digits=4)
report_dict = classification_report(y_true, y_pred, digits=4, output_dict=True)

print("Classification Report (Test Set):\n")
print(report_text)

report_txt_path = os.path.join(LOGS_DIR, "classification_report.txt")
report_json_path = os.path.join(LOGS_DIR, "classification_report.json")

with open(report_txt_path, "w") as f:
    f.write(report_text)

with open(report_json_path, "w") as f:
    json.dump(report_dict, f, indent=4)

cm = confusion_matrix(y_true, y_pred)
cm_dict = {"confusion_matrix": cm.tolist()}
cm_json_path = os.path.join(LOGS_DIR, "confusion_matrix_values.json")

with open(cm_json_path, "w") as f:
    json.dump(cm_dict, f, indent=4)

disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap="Blues", values_format="d")
plt.title("Confusion Matrix - DDR Baseline (Test Set)")

cm_plot_path = os.path.join(FIGURES_DIR, "confusion_matrix.png")
plt.savefig(cm_plot_path, dpi=300, bbox_inches="tight")
plt.show()
plt.close()

print("Saved:", report_txt_path)
print("Saved:", report_json_path)
print("Saved:", cm_json_path)
print("Saved:", cm_plot_path)

In [ ]:
# =========================
# CELL 20 — PER-CLASS METRICS + ROC CURVE
# =========================
precision_cls, recall_cls, f1_cls, support_cls = precision_recall_fscore_support(
    y_true, y_pred, labels=np.arange(NUM_CLASSES), zero_division=0
)

per_class_df = pd.DataFrame({
    "class": np.arange(NUM_CLASSES),
    "precision": precision_cls,
    "recall": recall_cls,
    "f1_score": f1_cls,
    "support": support_cls
})

per_class_csv = os.path.join(LOGS_DIR, "per_class_metrics.csv")
per_class_json = os.path.join(LOGS_DIR, "per_class_metrics.json")
per_class_df.to_csv(per_class_csv, index=False)
per_class_df.to_json(per_class_json, orient="records", indent=4)

y_true_bin = label_binarize(y_true, classes=np.arange(NUM_CLASSES))
roc_auc_per_class = {}

plt.figure(figsize=(8, 6))
for i in range(NUM_CLASSES):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_prob[:, i])
    roc_auc_val = auc(fpr, tpr)
    roc_auc_per_class[f"class_{i}"] = float(roc_auc_val)
    plt.plot(fpr, tpr, label=f"Class {i} (AUC = {roc_auc_val:.4f})")

plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - DDR Baseline")
plt.legend()
plt.tight_layout()

roc_curve_path = os.path.join(FIGURES_DIR, "roc_curve_baseline.png")
plt.savefig(roc_curve_path, dpi=300, bbox_inches="tight")
plt.show()
plt.close()

roc_auc_path = os.path.join(LOGS_DIR, "roc_auc_per_class.json")
with open(roc_auc_path, "w") as f:
    json.dump(roc_auc_per_class, f, indent=4)

print("Saved:", per_class_csv)
print("Saved:", per_class_json)
print("Saved:", roc_curve_path)
print("Saved:", roc_auc_path)

In [ ]:
# =========================
# CELL 21 — SAVE FINAL MODEL
# =========================
FINAL_MODEL_PATH = os.path.join(MODELS_DIR, "mobilenetv2_ddr_final.keras")
model.save(FINAL_MODEL_PATH)

print("Model saved to:", FINAL_MODEL_PATH)

In [ ]:
# =========================
# CELL 22 — SAVE RUN INFO + SUMMARY METRICS
# =========================
stage1_best_val_acc = max(history_stage1.history.get("val_accuracy", [None]))
stage1_best_val_loss = min(history_stage1.history.get("val_loss", [None]))

stage2_best_val_acc = max(history_stage2.history.get("val_accuracy", [None]))
stage2_best_val_loss = min(history_stage2.history.get("val_loss", [None]))

final_train_acc = history_stage2.history.get("accuracy", [None])[-1]
final_val_acc = history_stage2.history.get("val_accuracy", [None])[-1]
final_train_loss = history_stage2.history.get("loss", [None])[-1]
final_val_loss = history_stage2.history.get("val_loss", [None])[-1]

test_accuracy = accuracy_score(y_true, y_pred)
precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
    y_true, y_pred, average="macro", zero_division=0
)
precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
    y_true, y_pred, average="weighted", zero_division=0
)

y_true_bin = label_binarize(y_true, classes=np.arange(NUM_CLASSES))
auc_macro_ovr = roc_auc_score(y_true_bin, y_prob, multi_class="ovr", average="macro")
auc_weighted_ovr = roc_auc_score(y_true_bin, y_prob, multi_class="ovr", average="weighted")

summary_metrics = {
    "dataset_name": DATASET_NAME,
    "experiment_name": EXPERIMENT_NAME,
    "model_name": MODEL_NAME,
    "stage1_best_val_accuracy": None if stage1_best_val_acc is None else float(stage1_best_val_acc),
    "stage1_best_val_loss": None if stage1_best_val_loss is None else float(stage1_best_val_loss),
    "stage2_best_val_accuracy": None if stage2_best_val_acc is None else float(stage2_best_val_acc),
    "stage2_best_val_loss": None if stage2_best_val_loss is None else float(stage2_best_val_loss),
    "final_train_accuracy": None if final_train_acc is None else float(final_train_acc),
    "final_val_accuracy": None if final_val_acc is None else float(final_val_acc),
    "final_train_loss": None if final_train_loss is None else float(final_train_loss),
    "final_val_loss": None if final_val_loss is None else float(final_val_loss),
    "test_accuracy": float(test_accuracy),
    "test_precision_macro": float(precision_macro),
    "test_recall_macro": float(recall_macro),
    "test_f1_macro": float(f1_macro),
    "test_precision_weighted": float(precision_weighted),
    "test_recall_weighted": float(recall_weighted),
    "test_f1_weighted": float(f1_weighted),
    "test_auc_macro_ovr": float(auc_macro_ovr),
    "test_auc_weighted_ovr": float(auc_weighted_ovr)
}

summary_metrics_path = os.path.join(LOGS_DIR, "summary_metrics.json")
with open(summary_metrics_path, "w") as f:
    json.dump(summary_metrics, f, indent=4)

run_info = {
    "dataset_name": DATASET_NAME,
    "experiment_name": EXPERIMENT_NAME,
    "model_name": MODEL_NAME,
    "split_type": "global_train_val_test_split",
    "train_ratio": 0.70,
    "val_ratio": 0.15,
    "test_ratio": 0.15,
    "csv_path": CSV_PATH,
    "image_dir": IMAGE_DIR,
    "filename_column": filename_col,
    "label_column": label_col,
    "seed": SEED,
    "img_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
    "num_classes": int(NUM_CLASSES),
    "train_size": int(len(train_df)),
    "val_size": int(len(val_df)),
    "test_size": int(len(test_df)),
    "epochs_stage1": EPOCHS_STAGE1,
    "epochs_stage2": EPOCHS_STAGE2,
    "learning_rate_stage1": LEARNING_RATE_STAGE1,
    "learning_rate_stage2": LEARNING_RATE_STAGE2,
    "optimizer_stage1": OPTIMIZER_STAGE1,
    "optimizer_stage2": OPTIMIZER_STAGE2,
    "loss_function": LOSS_FUNCTION,
    "unfreeze_last_n_layers": UNFREEZE_LAST_N_LAYERS,
    "class_weights": class_weights,
    "artifacts": {
        "final_model_path": FINAL_MODEL_PATH,
        "history_stage1_path": history_stage1_path,
        "history_stage2_path": history_stage2_path,
        "history_combined_path": history_combined_path,
        "classification_report_txt": report_txt_path,
        "classification_report_json": report_json_path,
        "confusion_matrix_values": cm_json_path,
        "summary_metrics_path": summary_metrics_path,
        "training_history_figure": history_plot_path,
        "confusion_matrix_figure": cm_plot_path,
        "roc_curve_figure": roc_curve_path,
        "per_class_metrics_csv": per_class_csv,
        "per_class_metrics_json": per_class_json
    }
}

run_info_path = os.path.join(LOGS_DIR, "mobilenetv2_ddr_run_info.json")
with open(run_info_path, "w") as f:
    json.dump(run_info, f, indent=4)

print("Saved:", summary_metrics_path)
print("Saved:", run_info_path)

In [ ]:
# =========================
# CELL 23 — QUICK RESULT SUMMARY
# =========================
with open(os.path.join(LOGS_DIR, "summary_metrics.json"), "r") as f:
    metrics = json.load(f)

print(json.dumps(metrics, indent=4))